In [1]:
# Measure the time it takes every cell to run
%pip install ipython-autotime
%load_ext autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 24.2 MB/s eta 0:00:00
time: 300 µs (started: 2026-04-20 17:31:17 +00:00)


In [2]:
# Set up idc-index, used to make clinical table parameters human-readable
%pip install idc-index
from idc_index import IDCClient
c = IDCClient()
c.fetch_index('clinical_index')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.1/74.1 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 40.8 MB/s eta 0:00:00
time: 25.3 s (started: 2026-04-20 17:31:22 +00:00)


In [3]:
# Set up BigQuery, used to construct patient cohort of interest
import os
my_ProjectID = "nlst-radiomics"
os.environ["GCP_PROJECT_ID"] = my_ProjectID

from google.colab import auth, files
auth.authenticate_user()

from google.cloud import bigquery
bq_client = bigquery.Client(my_ProjectID)

time: 12.7 s (started: 2026-04-20 17:32:33 +00:00)


In [4]:
# First, what are the possible segmentation types?
# Note: I checked that 'AIMI lung and nodule AI segmentation' contains 'Lung' and 'Nodule' segment labels

seg_query = """
SELECT SeriesDescription
FROM `bigquery-public-data.idc_v23.dicom_all`
WHERE
  collection_id = 'nlst'
  AND Modality = 'SEG'
"""

seg_result = bq_client.query(seg_query)
seg_metadata_df = seg_result.result().to_dataframe()

print(f"\n--- Unique Series Descriptions ---")
display(seg_metadata_df['SeriesDescription'].value_counts().head(20))

print(f"\n--- Unique Series Descriptions Containing 'lung and nodule' ---")
lung_nodule_segmentation_types = seg_metadata_df[seg_metadata_df['SeriesDescription'].str.contains('lung and nodule', case=False)]['SeriesDescription'].unique().tolist()
display(lung_nodule_segmentation_types)


--- Unique Series Descriptions ---


,count
SeriesDescription,
TotalSegmentator(v1.5.6) Segmentation of Series 2,47987
TotalSegmentator(v1.5.6) Segmentation of Series 3,38198
TotalSegmentator(v1.5.6) Segmentation of Series 4,13362
TotalSegmentator(v1.5.6) Segmentation of Series 5,6180
TotalSegmentator(v1.5.6) Segmentation of Series 6,5377
TotalSegmentator(v1.5.6) Segmentation of Series 1,4449
TotalSegmentator(v1.5.6) Segmentation of Series 0,1132
AIMI lung and nodule AI segmentation,1042
3d_fullres-tta_nnU-Net_Segmentation,1039



--- Unique Series Descriptions Containing 'lung and nodule' ---


['AIMI lung and nodule radiologist 4 corrected segmentation',
 'AIMI lung and nodule radiologist 5 corrected segmentation',
 'AIMI lung and nodule radiologist 8 corrected segmentation',
 'AIMI lung and nodule AI segmentation']

time: 4.66 s (started: 2026-04-20 17:32:50 +00:00)


In [5]:
# How many CT/SEG pairs with lung and nodule segmentations are there?
# Note: If seg.SeriesDescription is not radiologist corrected, there are 1042 SEG files with lung and nodule segmentations

ct_seg_join_query = f"""
SELECT
  DISTINCT seg.SeriesInstanceUID AS SEG_SeriesInstanceUID,
  ct.PatientID,
  ct.SeriesInstanceUID AS CT_SeriesInstanceUID,
  seg.SeriesDescription AS SEG_SeriesDescription,
  seg.StudyDate,
  CASE
    WHEN FORMAT_DATE('%Y', seg.StudyDate) LIKE '1999%' THEN 'T0'
    WHEN FORMAT_DATE('%Y', seg.StudyDate) LIKE '2000%' THEN 'T1'
    WHEN FORMAT_DATE('%Y', seg.StudyDate) LIKE '2001%' THEN 'T2'
    ELSE 'Other'
  END AS StudyYear
FROM
  `bigquery-public-data.idc_v23.dicom_all` AS ct
JOIN
  `bigquery-public-data.idc_v23.dicom_all` AS seg
ON
  ct.collection_id = seg.collection_id
  AND ct.PatientID = seg.PatientID
WHERE
  ct.collection_id = 'nlst'
  AND ct.Modality = 'CT'
  AND seg.Modality = 'SEG'
  AND seg.SeriesDescription IN ({', '.join([f"'{s}'" for s in lung_nodule_segmentation_types])})
  AND EXISTS (
    SELECT 1
    FROM UNNEST(seg.ReferencedSeriesSequence) AS ref_seq
    WHERE ct.SeriesInstanceUID = ref_seq.SeriesInstanceUID
  )
"""

ct_seg_join_result = bq_client.query(ct_seg_join_query)
ct_seg_join_df = ct_seg_join_result.result().to_dataframe()

print(f"Number of distinct patients with a CT/SEG pair: {len(ct_seg_join_df['PatientID'].unique())}")
print(f"Number of unique CT series: {len(ct_seg_join_df['CT_SeriesInstanceUID'].unique())}")
print(f"Number of CT/SEG pairs: {ct_seg_join_df.shape[0]}")
print(f"Number of distinct study dates: {len(ct_seg_join_df['StudyDate'].unique())}")

Number of distinct patients with a CT/SEG pair: 572
Number of unique CT series: 1042
Number of CT/SEG pairs: 1144
Number of distinct study dates: 3
time: 4.12 s (started: 2026-04-20 17:32:58 +00:00)


In [6]:
# Two SEG series can correspond to one CT series because of radiologist corrected segmentations. "Keep" the CT/SEG pair with the radiologist corrected segmentation and throw out the other.

priority_segmentation_type = 'AIMI lung and nodule AI segmentation'

# Create a priority column: 0 for non-AI, 1 for AI
ct_seg_join_df['priority'] = ct_seg_join_df['SEG_SeriesDescription'].apply(lambda x: 1 if x == priority_segmentation_type else 0)

# Sort by CT_SeriesInstanceUID and then by priority (0 comes before 1, so non-AI is prioritized)
ct_seg_join_df_filtered = ct_seg_join_df.sort_values(by=['CT_SeriesInstanceUID', 'priority'])

# Drop duplicates, keeping the first (which will be the non-AI one if present, otherwise the first AI one)
ct_seg_join_df_one_to_one = ct_seg_join_df_filtered.drop_duplicates(subset='CT_SeriesInstanceUID', keep='first')

print(f"\nNumber of CT/SEG pairs after filtering: {ct_seg_join_df_one_to_one.shape[0]}")
display(ct_seg_join_df_one_to_one.head())


Number of CT/SEG pairs after filtering: 1042


,SEG_SeriesInstanceUID,PatientID,CT_SeriesInstanceUID,SEG_SeriesDescription,StudyDate,StudyYear,priority
494,1.2.276.0.7230010.3.1.3.17436516.3112497.17206...,107002,1.2.840.113654.2.55.10057224705584082952969157...,AIMI lung and nodule AI segmentation,1999-01-02,T0,1
6,1.2.276.0.7230010.3.1.3.17436516.3112767.17206...,107030,1.2.840.113654.2.55.10087518978221069034420730...,AIMI lung and nodule AI segmentation,1999-01-02,T0,1
544,1.2.276.0.7230010.3.1.3.17436516.3182127.17206...,126955,1.2.840.113654.2.55.10212803674068251180698456...,AIMI lung and nodule AI segmentation,2000-01-02,T1,1
909,1.2.276.0.7230010.3.1.3.17436516.3103212.17206...,104999,1.2.840.113654.2.55.10222078521568314836955069...,AIMI lung and nodule AI segmentation,1999-01-02,T0,1
614,1.2.276.0.7230010.3.1.3.17436516.3180509.17206...,126446,1.2.840.113654.2.55.10223655654547815709262155...,AIMI lung and nodule radiologist 5 corrected s...,1999-01-02,T0,0


time: 49.5 ms (started: 2026-04-20 17:33:06 +00:00)


In [7]:
# Download CT/SEG pairs for 5 distinct patients using idc_index

download_dir_on_colab_vm = "/content/malignant_trial"
if not os.path.exists(download_dir_on_colab_vm):
    os.makedirs(download_dir_on_colab_vm)

malignant_patientIDs = ct_seg_join_df_one_to_one['PatientID'].unique().tolist()

five_malignant_patientIDs = malignant_patientIDs[:5]
CT_SEG_seriesIDs = ct_seg_join_df_one_to_one[ct_seg_join_df_one_to_one['PatientID'].isin(five_malignant_patientIDs)][['CT_SeriesInstanceUID', 'SEG_SeriesInstanceUID']].values.flatten().tolist()

print(f"Downloading {len(CT_SEG_seriesIDs)} series for {len(five_malignant_patientIDs)} patients to Colab VM at {download_dir_on_colab_vm}...")
c.download_from_selection(seriesInstanceUID=CT_SEG_seriesIDs, downloadDir=download_dir_on_colab_vm)
print("Download to Colab VM complete.")

Download to Colab VM complete.
time: 10.6 s (started: 2026-04-20 17:33:08 +00:00)


In [ ]:
# Compress the downloaded directory into a zip file
#zip_filename = 'malignant_trial_data.zip'
#!zip -r "{zip_filename}" "{download_dir_on_colab_vm}"

# Trigger download to your local machine
#print(f"\nTriggering download of '{zip_filename}' to your local machine...")
#files.download(zip_filename)
#print("Download initiated. Please check your browser's downloads.")

time: 458 µs (started: 2026-04-07 15:07:02 +00:00)


In [8]:
# first figure out how to convert CT file into a .nrrd file
%pip install SimpleITK
import SimpleITK as sitk
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 6.0 MB/s eta 0:00:00
time: 15.8 s (started: 2026-04-20 17:33:22 +00:00)


In [9]:
def convert_CT_to_nrrd(input_folder, output_folder):
  # assume input_folder and output_folder exist; case handling elsewhere
  reader = sitk.ImageSeriesReader()
  dicom_names = reader.GetGDCMSeriesFileNames(input_folder)
  reader.SetFileNames(dicom_names)
  ct_image = reader.Execute()
  sitk.WriteImage(ct_image, os.path.join(output_folder, 'CT.nrrd'))

time: 4.33 ms (started: 2026-04-20 17:33:41 +00:00)


In [10]:
%pip install dcmqi
import subprocess
import json
import glob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 6.3 MB/s eta 0:00:00
time: 16.7 s (started: 2026-04-20 17:33:43 +00:00)


In [11]:
def convert_SEG_to_nrrd(input_file, output_folder, prefix="SEG"):
  command = ["segimage2itkimage", "--inputDICOM", input_file, "--outputDirectory", output_folder, "--outputType", "nrrd", "--prefix", prefix]
  try:
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    return True
  except subprocess.CalledProcessError as e:
    print(f"Conversion of SEG to .nrrd failed: {e}")
    return False

time: 2.59 ms (started: 2026-04-20 17:34:08 +00:00)


In [12]:
def identify_and_rename_masks_assuming_naming_convention(timestep_path, prefix="SEG"):
  os.rename(os.path.join(timestep_path, f"{prefix}-1.nrrd"), os.path.join(timestep_path, "lung.nrrd"))
  os.rename(os.path.join(timestep_path, f"{prefix}-2.nrrd"), os.path.join(timestep_path, "nodule.nrrd"))
  os.remove(os.path.join(timestep_path, f"{prefix}-meta.json"))

time: 1.05 ms (started: 2026-04-20 17:34:11 +00:00)


In [13]:
def resample_isotropic_and_align(timestep_path, target_spacing=[1.0,1.0,1.0]):
  ct = sitk.ReadImage(os.path.join(timestep_path, "CT.nrrd"))
  lung_mask = sitk.ReadImage(os.path.join(timestep_path, "lung.nrrd"))
  nodule_mask = sitk.ReadImage(os.path.join(timestep_path, "nodule.nrrd"))

  original_spacing = ct.GetSpacing()
  original_size = ct.GetSize()
  new_size = [int(round(original_size[i] * (original_spacing[i] / target_spacing[i]))) for i in range(3)]

  resampler = sitk.ResampleImageFilter()
  resampler.SetOutputSpacing(target_spacing)
  resampler.SetSize(new_size)
  resampler.SetOutputDirection(ct.GetDirection())
  resampler.SetOutputOrigin(ct.GetOrigin())
  resampler.SetInterpolator(sitk.sitkLanczosWindowedSinc)

  isotropic_ct = resampler.Execute(ct)

  resampler.SetReferenceImage(isotropic_ct)
  resampler.SetInterpolator(sitk.sitkNearestNeighbor)
  aligned_lung_mask = resampler.Execute(lung_mask)
  aligned_nodule_mask = resampler.Execute(nodule_mask)

  sitk.WriteImage(isotropic_ct, os.path.join(timestep_path, "CT_iso.nrrd"))
  sitk.WriteImage(aligned_lung_mask, os.path.join(timestep_path, "lung_iso.nrrd"))
  sitk.WriteImage(aligned_nodule_mask, os.path.join(timestep_path, "nodule_iso.nrrd"))

time: 3.66 ms (started: 2026-04-20 17:34:13 +00:00)


In [14]:
def align_masks(timestep_path):
  ct = sitk.ReadImage(os.path.join(timestep_path, "CT.nrrd"))
  lung_mask = sitk.ReadImage(os.path.join(timestep_path, "lung.nrrd"))
  nodule_mask = sitk.ReadImage(os.path.join(timestep_path, "nodule.nrrd"))
  resampler = sitk.ResampleImageFilter()
  resampler.SetReferenceImage(ct)
  resampler.SetInterpolator(sitk.sitkNearestNeighbor)
  aligned_lung_mask = resampler.Execute(lung_mask)
  aligned_nodule_mask = resampler.Execute(nodule_mask)
  sitk.WriteImage(aligned_lung_mask, os.path.join(timestep_path, "lung_aligned.nrrd"))
  sitk.WriteImage(aligned_nodule_mask, os.path.join(timestep_path, "nodule_aligned.nrrd"))

time: 1.18 ms (started: 2026-04-20 17:34:16 +00:00)


In [15]:
%pip install pynrrd
import nrrd
import numpy as np
import matplotlib.pyplot as plt

time: 6.96 s (started: 2026-04-20 17:34:18 +00:00)


In [16]:
def plot_ct_with_masks(patient_id, timestep_dir, timestep_path, suffix="iso"):
  if suffix == "iso":
    ct, _ = nrrd.read(os.path.join(timestep_path, f"CT_iso.nrrd"))
  else:
    ct, _ = nrrd.read(os.path.join(timestep_path, f"CT.nrrd"))
  lung, _ = nrrd.read(os.path.join(timestep_path, f"lung_{suffix}.nrrd"))
  nodule, _ = nrrd.read(os.path.join(timestep_path, f"nodule_{suffix}.nrrd"))

  slice_counts = np.sum(nodule > 0, axis=(0,1))
  z_slice = np.argmax(slice_counts)
  ct_slice = ct[:, :, z_slice]
  lung_slice = lung[:, :, z_slice]
  nodule_slice = nodule[:, :, z_slice]

  fig, axes = plt.subplots(2,2,figsize=(12,12))
  axes[0,0].imshow(ct_slice, cmap='gray', vmin=-1000, vmax=400, origin='lower')
  axes[0,0].set_title("Raw CT Slice")
  axes[0,0].axis('off')

  axes[0,1].imshow(ct_slice, cmap='gray', vmin=-1000, vmax=400, origin='lower')
  lung_mask = np.ma.masked_where(lung_slice == 0, lung_slice)
  axes[0,1].imshow(lung_mask, cmap='Blues', alpha=0.3, origin='lower', vmin=0, vmax=1)
  nodule_mask = np.ma.masked_where(nodule_slice == 0, nodule_slice)
  axes[0,1].imshow(nodule_mask, cmap='Reds', alpha=0.6, origin='lower', vmin=0, vmax=1)
  axes[0,1].set_title("CT Slice with Lung and Nodule Masks")
  axes[0,1].axis('off')

  axes[1,0].imshow(nodule_slice, cmap='gray', origin='lower')
  axes[1,0].set_title("Nodule Mask Only")
  axes[1,0].axis('off')

  nodule_voxels = ct_slice[nodule_slice > 0]
  axes[1,1].hist(nodule_voxels, bins=30, color='crimson', edgecolor='black')
  axes[1,1].set_title("Nodule HU Histogram")
  axes[1,1].set_xlabel("Hounsfield Units (HU)")
  axes[1,1].set_ylabel("Voxel Count")

  fig.suptitle(f"Patient ID: {patient_id}, Timestep: {timestep_dir}")
  plt.tight_layout()
  plt.savefig(os.path.join(timestep_path, f"CT_with_masks_{suffix}.png"), dpi=300, bbox_inches='tight')
  plt.close(fig)

time: 5.88 ms (started: 2026-04-20 17:34:28 +00:00)


In [17]:
def prepare_for_pyradiomics(base_path, resample=False, plot=False):
  for patient_id in os.listdir(base_path):
    patient_path = os.path.join(base_path, patient_id)
    for timestep_dir in os.listdir(patient_path):
        timestep_path = os.path.join(patient_path, timestep_dir)
        for item in os.listdir(timestep_path):
            full_item_path = os.path.join(timestep_path, item)
            if os.path.isdir(full_item_path):
                if item.startswith('CT_'):
                    ct_series_path = full_item_path
                elif item.startswith('SEG_'):
                    seg_series_path = full_item_path
        try:
          convert_CT_to_nrrd(ct_series_path, timestep_path)
          print(f"Converted CT for patient {patient_id}, timestep {timestep_dir} to {timestep_path}")

          seg_dcm_files = [f for f in os.listdir(seg_series_path) if f.endswith('.dcm')]
          seg_file = os.path.join(seg_series_path, seg_dcm_files[0])
          convert_SEG_to_nrrd(seg_file, timestep_path)
          print(f"Converted SEG for patient {patient_id}, timestep {timestep_dir} to {timestep_dir}")

          identify_and_rename_masks_assuming_naming_convention(timestep_path)
          print(f"Renamed masks for patient {patient_id}, timestep {timestep_dir}")

          if resample:
            resample_isotropic_and_align(timestep_path)
            print(f"Resampled and aligned CT and masks for patient {patient_id}, timestep {timestep_dir} to {timestep_dir}")
            if plot:
              plot_ct_with_masks(patient_id, timestep_dir, timestep_path)
              print(f"Plotted CT with masks for patient {patient_id}, timestep {timestep_dir}")
          else:
            if plot:
              align_masks(timestep_path)
              print(f"Aligned masks with CT for patient {patient_id}, timestep {timestep_dir} to {timestep_dir}")
              plot_ct_with_masks(patient_id, timestep_dir, timestep_path, suffix="aligned")
              print(f"Plotted CT with masks for patient {patient_id}, timestep {timestep_dir}")
        except Exception as e:
                print(f"Failed for patient {patient_id}, timestep {timestep_dir}: {e}")

time: 6.33 ms (started: 2026-04-20 17:34:31 +00:00)


In [18]:
%pip install git+https://github.com/Radiomics/pyradiomics.git

from radiomics import featureextractor
import pandas as pd

  Cloning https://github.com/Radiomics/pyradiomics.git to /tmp/pip-req-build-otxfa3j2
  Running command git clone --filter=blob:none --quiet https://github.com/Radiomics/pyradiomics.git /tmp/pip-req-build-otxfa3j2
  Resolved https://github.com/Radiomics/pyradiomics.git to commit 8ed579383b44806651c463d5e691f3b2b57522ab
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 6.2 MB/s eta 0:00:00
  Created wheel for pyradiomics: filename=pyradiomics-3.1.1.dev111+g8ed579383-cp312-cp312-linux_x86_64.whl size=121732 sha256=7a95208708095b158e8b47b92330e441173f3a8343ed89852611a88a62791e58
  Stored in directory: /tmp/pip-ephem-wheel-cache-janbqzwb/wheels/33/e9/c0/7de3e16cb600bae494d4a94dcd6c30a0443f06fb7359e87aa3
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-

In [19]:
from radiomics.featureextractor import RadiomicsFeatureExtractor, _SingletonGeometryTolerance

from __future__ import annotations

import collections
import json
import logging
import os
import pathlib
import threading
from itertools import chain

import pykwalify.core
import SimpleITK as sitk

from radiomics import (
    generalinfo,
    getFeatureClasses,
    getImageTypes,
    getParameterValidationFiles,
    imageoperations,
)

logger = logging.getLogger(__name__)

class SaveCropFeatureExtractor(RadiomicsFeatureExtractor):
    def execute(
        self,
        imageFilepath,
        maskFilepath,
        label=None,
        label_channel=None,
        voxelBased=False,
    ):
        """
        Compute radiomics signature for provide image and mask combination. It comprises of the following steps:

        1. Image and mask are loaded and normalized/resampled if necessary.
        2. Validity of ROI is checked using :py:func:`~imageoperations.checkMask`, which also computes and returns the
           bounding box.
        3. If enabled, provenance information is calculated and stored as part of the result. (Not available in voxel-based
           extraction)
        4. Shape features are calculated on a cropped (no padding) version of the original image. (Not available in
           voxel-based extraction)
        5. If enabled, resegment the mask based upon the range specified in ``resegmentRange`` (default None: resegmentation
           disabled).
        6. Other enabled feature classes are calculated using all specified image types in ``_enabledImageTypes``. Images
           are cropped to tumor mask (no padding) after application of any filter and before being passed to the feature
           class.
        7. The calculated features is returned as ``collections.OrderedDict``.

        :param imageFilepath: SimpleITK Image, or string pointing to image file location
        :param maskFilepath: SimpleITK Image, or string pointing to labelmap file location
        :param label: Integer, value of the label for which to extract features. If not specified, last specified label
            is used. Default label is 1.
        :param label_channel: Integer, index of the channel to use when maskFilepath yields a SimpleITK.Image with a vector
            pixel type. Default index is 0.
        :param voxelBased: Boolean, default False. If set to true, a voxel-based extraction is performed, segment-based
            otherwise.
        :returns: dictionary containing calculated signature ("<imageType>_<featureClass>_<featureName>":value).
            In case of segment-based extraction, value type for features is float, if voxel-based, type is SimpleITK.Image.
            Type of diagnostic features differs, but can always be represented as a string.
        """
        _settings = self.settings.copy()

        tolerance = _settings.get("geometryTolerance")
        additionalInfo = _settings.get("additionalInfo", False)
        resegmentShape = _settings.get("resegmentShape", False)

        if label is not None:
            _settings["label"] = label
        else:
            label = _settings.get("label", 1)

        if label_channel is not None:
            _settings["label_channel"] = label_channel

        if _SingletonGeometryTolerance().geometryTolerance != tolerance:
            self._setTolerance()

        if additionalInfo:
            generalInfo = generalinfo.GeneralInfo()
            generalInfo.addGeneralSettings(_settings)
            generalInfo.addEnabledImageTypes(self.enabledImagetypes)
        else:
            generalInfo = None

        if voxelBased:
            _settings["voxelBased"] = True
            kernelRadius = _settings.get("kernelRadius", 1)
            logger.info("Starting voxel based extraction")
        else:
            kernelRadius = 0

        logger.info("Calculating features with label: %d", label)
        logger.debug("Enabled images types: %s", self.enabledImagetypes)
        logger.debug("Enabled features: %s", self.enabledFeatures)
        logger.debug("Current settings: %s", _settings)

        # 1. Load the image and mask
        featureVector = collections.OrderedDict()
        image, mask = self.loadImage(
            imageFilepath, maskFilepath, generalInfo, **_settings
        )

        # 2. Check whether loaded mask contains a valid ROI for feature extraction and get bounding box
        # Raises a ValueError if the ROI is invalid
        boundingBox, correctedMask = imageoperations.checkMask(image, mask, **_settings)

        # Update the mask if it had to be resampled
        if correctedMask is not None:
            if generalInfo is not None:
                generalInfo.addMaskElements(image, correctedMask, label, "corrected")
            mask = correctedMask

        logger.debug("Image and Mask loaded and valid, starting extraction")

        # 5. Resegment the mask if enabled (parameter regsegmentMask is not None)
        resegmentedMask = None
        if _settings.get("resegmentRange", None) is not None:
            resegmentedMask = imageoperations.resegmentMask(image, mask, **_settings)

            # Recheck to see if the mask is still valid, raises a ValueError if not
            boundingBox, correctedMask = imageoperations.checkMask(
                image, resegmentedMask, **_settings
            )

            if generalInfo is not None:
                generalInfo.addMaskElements(
                    image, resegmentedMask, label, "resegmented"
                )

        # 3. Add the additional information if enabled
        if generalInfo is not None:
            featureVector.update(generalInfo.getGeneralInfo())

        # if resegmentShape is True and resegmentation has been enabled, update the mask here to also use the
        # resegmented mask for shape calculation (e.g. PET resegmentation)
        if resegmentShape and resegmentedMask is not None:
            mask = resegmentedMask

        if not voxelBased:
            # 4. If shape descriptors should be calculated, handle it separately here
            featureVector.update(
                self.computeShape(image, mask, boundingBox, **_settings)
            )

        # (Default) Only use resegemented mask for feature classes other than shape
        # can be overridden by specifying `resegmentShape` = True
        if not resegmentShape and resegmentedMask is not None:
            mask = resegmentedMask

        # 6. Calculate other enabled feature classes using enabled image types
        # Make generators for all enabled image types
        logger.debug("Creating image type iterator")
        imageGenerators = []
        for imageType, customKwargs in self.enabledImagetypes.items():
            args = _settings.copy()
            args.update(customKwargs)
            msg = f'Adding image type "{imageType}" with custom settings: {customKwargs!s}'
            logger.info(msg)
            imageGenerators = chain(
                imageGenerators,
                getattr(imageoperations, f"get{imageType}Image")(image, mask, **args),
            )

        logger.debug("Extracting features")
        # Calculate features for all (filtered) images in the generator
        for originputImage, imageTypeName, inputKwargs in imageGenerators:
            logger.info("Calculating features for %s image", imageTypeName)
            inputImage, inputMask = imageoperations.cropToTumorMask(
                originputImage, mask, boundingBox, padDistance=kernelRadius
            )

            if isinstance(maskFilepath, str):
                sitk.WriteImage(sitk.Cast(inputMask, sitk.sitkUInt8), maskFilepath[:-5] + "_cropped.nrrd", useCompression=True)
                print("Saving cropped nodule .nrrd file")

            featureVector.update(
                self.computeFeatures(
                    inputImage, inputMask, imageTypeName, **inputKwargs
                )
            )

        logger.debug("Features extracted")

        return featureVector

time: 24.5 ms (started: 2026-04-20 17:35:24 +00:00)


In [20]:
def extract_features(base_path, out_path, resample=True, new_ps=[1.0,1.0,1.0], interpolator='sitkLanczosWindowedSinc'):
  settings = {}
  settings['resegmentRange'] = [-1000, 2000] # Excludes air and bone (HU scale) for feature calculation
  settings['label'] = 2 # Assumes nodule is label 2
  if resample:
    ct_name = 'CT'
    mask_name = 'nodule'
    settings['interpolator'] = interpolator
    settings['resampledPixelSpacing'] = new_ps
    print(f'Isotropic resampling to {new_ps} using interpolator {interpolator}')
  else:
    ct_name = 'CT_iso'
    mask_name = 'nodule_iso'

  print('Extracting radiomic features...')
  extractor = SaveCropFeatureExtractor(**settings)

  dataFrames = []
  for patient in os.listdir(base_path):
      patient_path = os.path.join(base_path, patient)
      if not os.path.isdir(patient_path):
          continue

      for timestep in os.listdir(patient_path):
          timestep_path = os.path.join(patient_path, timestep)
          mask_path = os.path.join(timestep_path, mask_name + '.nrrd')
          ct_path = os.path.join(timestep_path, ct_name + '.nrrd')

          if not (os.path.exists(ct_path) and os.path.exists(mask_path)):
              print(f"Skipping {patient}/{timestep}: CT or mask not found.")
              continue

          try:
              features = extractor.execute(ct_path, mask_path)
              df = pd.DataFrame.from_dict({i:str(features[i]) for i in features}, orient='index', columns=[patient+timestep])
              dataFrames.append(df)
              print(f'Extracted features from {patient}/{timestep}')
          except Exception as e:
              print(f"Failed to extract features from {patient}/{timestep}: {e}")

  if dataFrames:
      data = pd.concat(dataFrames, axis=1)
      data.to_csv(out_path)
      print(f'Saved radiomic features to {out_path}')
  else:
      print("No features were extracted.")

time: 3.58 ms (started: 2026-04-20 17:35:35 +00:00)


In [21]:
# pyradiomics does the resampling, skip plots
prepare_for_pyradiomics('/content/malignant_trial/nlst', plot=True)

Converted CT for patient 126955, timestep 1.2.840.113654.2.55.19724172053671692532719125746870801108 to /content/malignant_trial/nlst/126955/1.2.840.113654.2.55.19724172053671692532719125746870801108
Converted SEG for patient 126955, timestep 1.2.840.113654.2.55.19724172053671692532719125746870801108 to 1.2.840.113654.2.55.19724172053671692532719125746870801108
Renamed masks for patient 126955, timestep 1.2.840.113654.2.55.19724172053671692532719125746870801108
Aligned masks with CT for patient 126955, timestep 1.2.840.113654.2.55.19724172053671692532719125746870801108 to 1.2.840.113654.2.55.19724172053671692532719125746870801108
Plotted CT with masks for patient 126955, timestep 1.2.840.113654.2.55.19724172053671692532719125746870801108
Converted CT for patient 126955, timestep 1.2.840.113654.2.55.245452063932624888085703626412362172760 to /content/malignant_trial/nlst/126955/1.2.840.113654.2.55.245452063932624888085703626412362172760
Converted SEG for patient 126955, timestep 1.2.840

In [ ]:
base_path = '/content/malignant_trial/nlst'
out_path = '/content/malignant_trial/features_pyradiomics_resampling.csv'
extract_features(base_path, out_path, interpolator="sitkBSpline")

INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'resegmentRange': [-1000, 2000], 'label': 2, 'interpolator': 'sitkBSpline', 'resampledPixelSpacing': [1.0, 1.0, 1.0]}
INFO:radiomics.featureextractor:Loading image and mask


Isotropic resampling to [1.0, 1.0, 1.0] using interpolator sitkBSpline
Extracting radiomic features...


INFO:radiomics.imageoperations:Applying resampling from spacing [0.603516 0.603516 2.5     ] and size [512 512 115] to spacing [1.  1.  2.5] and size [16, 17, 13]
INFO:radiomics.featureextractor:Computing shape
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Loading image and mask


Saving cropped nodule .nrrd file
Extracted features from 107002/1.2.840.113654.2.55.136072716603498717401162853674652254104


INFO:radiomics.imageoperations:Applying resampling from spacing [0.605469 0.605469 2.5     ] and size [512 512  99] to spacing [1. 1. 1.] and size [250, 128, 97]
INFO:radiomics.featureextractor:Computing shape
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


...

Saving cropped nodule .nrrd file
Extracted features from 126955/1.2.840.113654.2.55.245452063932624888085703626412362172760


INFO:radiomics.imageoperations:Applying resampling from spacing [0.78125 0.78125 2.     ] and size [512 512 118] to spacing [1. 1. 1.] and size [271, 219, 211]
INFO:radiomics.featureextractor:Computing shape
INFO:radiomics.featureextractor:Computing firstorder


Saving cropped nodule .nrrd file


INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm


Extracted features from 126446/1.2.840.113654.2.55.257572809223995827394443746663383442894
Saved radiomic features to /content/malignant_trial/features_pyradiomics_resampling.csv
time: 3min 15s (started: 2026-04-07 15:09:43 +00:00)


In [ ]:
%pip install pydicom
import pydicom
import csv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.8 MB/s eta 0:00:00
time: 6.47 s (started: 2026-04-07 15:13:27 +00:00)


In [ ]:
# Adapted dicom_header.py to extract header info from CT headers

def extract_ct_header_info(patients_dir, csv_filepath):
    patients = os.listdir(patients_dir)
    headerinfo = {}
    headers = ["Manufacturer","ManufacturerModelName","SliceThickness","KVP","DataCollectionDiameter","FilterType","FocalSpots","ConvolutionKernel","ExposureTime","XRayTubeCurrent","Exposure", "PixelSpacing"]
    for p in patients:
        patient_path = os.path.join(patients_dir, p)
        if not os.path.isdir(patient_path):
            continue # Skip if not a directory

        timesteps = os.listdir(patient_path)
        for timestep in timesteps:
            timestep_path = os.path.join(patient_path, timestep)
            if not os.path.isdir(timestep_path):
                continue # Skip if not a directory

            ct_info = {}
            print(f"Extracting CT header info from {p}/{timestep}")
            ct_series_dir = None
            for item in os.listdir(timestep_path):
                full_item_path = os.path.join(timestep_path, item)
                if os.path.isdir(full_item_path) and item.startswith('CT_'):
                    ct_series_dir = full_item_path
                    break # Found the CT series directory

            if ct_series_dir:
                # Now, look for a DICOM file inside the CT series directory
                dicom_files = [f for f in os.listdir(ct_series_dir) if f.endswith('.dcm')]
                if dicom_files:
                    ct_file_path = os.path.join(ct_series_dir, dicom_files[0]) # Take the first one
                    try:
                        ct_img = pydicom.dcmread(ct_file_path)
                        for header in headers:
                            key = header
                            value = ct_img.get(header, None)
                            if value is not None:
                                ct_info[key] = value
                            else:
                                ct_info[key] = "N/A"
                        headerinfo[p+"_"+timestep] = ct_info
                    except Exception as e:
                        print(f"Failed to read DICOM file {ct_file_path}: {e}")
                else:
                    print(f"No DICOM files found in {ct_series_dir}")
            else:
                print(f"CT series directory not found in {timestep_path}")

    headers.insert(0,"Timestep")
    file_exists = os.path.exists(csv_filepath)
    with open(csv_filepath, "a", newline='') as f: # Changed mode to 'a' for append
        w = csv.DictWriter(f, headers)
        if not file_exists: # Write header only if file does not exist
            w.writeheader()
        for key,val in sorted(headerinfo.items()):
            row = {'Timestep': key}
            row.update(val)
            w.writerow(row)
    print(f'Saved CT header info to {csv_filepath}')

time: 6.9 ms (started: 2026-04-07 15:32:36 +00:00)


In [ ]:
patients_dir = "/content/malignant_trial/nlst"
csv_filepath = "/content/malignant_trial/ct_header_info.csv"
extract_ct_header_info(patients_dir, csv_filepath)

Reading CT from 107002/1.2.840.113654.2.55.136072716603498717401162853674652254104
Reading CT from 107002/1.2.840.113654.2.55.35671975265195223904685112987162619590
Reading CT from 107030/1.2.840.113654.2.55.279742797608242185321251439412013425352
Reading CT from 107030/1.2.840.113654.2.55.75126321162854904399604743122014916460
Reading CT from 104999/1.2.840.113654.2.55.105535245484324310535029058008342584740
Reading CT from 104999/1.2.840.113654.2.55.77681748244476164967788827234616069630
Reading CT from 104999/1.2.840.113654.2.55.168698666181948219239364956246822360078
Reading CT from 126955/1.2.840.113654.2.55.19724172053671692532719125746870801108
Reading CT from 126955/1.2.840.113654.2.55.245452063932624888085703626412362172760
Reading CT from 126446/1.2.840.113654.2.55.257572809223995827394443746663383442894
Saved CT header info to /content/malignant_trial/ct_header_info.csv
time: 69.3 ms (started: 2026-04-07 15:13:43 +00:00)


In [ ]:
import shutil

# takes in indices to use on malignant_patients list
def download_data(download_dir, lower_index, higher_index):
    patientIDs = malignant_patientIDs[lower_index:higher_index]
    CT_SEG_seriesIDs = ct_seg_join_df_one_to_one[ct_seg_join_df_one_to_one['PatientID'].isin(patientIDs)][['CT_SeriesInstanceUID', 'SEG_SeriesInstanceUID']].values.flatten().tolist()
    c.download_from_selection(seriesInstanceUID=CT_SEG_seriesIDs, downloadDir=download_dir)
    print(f"Downloaded {len(CT_SEG_seriesIDs)} series for {len(patientIDs)} patients to Colab VM at {download_dir}...")

def delete_data(base_path):
  for patient_id in os.listdir(base_path):
    patient_path = os.path.join(base_path, patient_id)
    if not os.path.isdir(patient_path):
        continue

    for timestep_dir in os.listdir(patient_path):
        timestep_path = os.path.join(patient_path, timestep_dir)
        if not os.path.isdir(timestep_path):
            continue

        for item in os.listdir(timestep_path):
            full_item_path = os.path.join(timestep_path, item)
            if os.path.isdir(full_item_path) and (item.startswith('CT_') or item.startswith('SEG_')):
                try:
                    shutil.rmtree(full_item_path)
                except Exception as e:
                    print(f"Failed to delete {full_item_path}: {e}")
        print(f"Deleted CT and SEG series data for patient {patient_id}")

time: 5.59 ms (started: 2026-04-07 15:32:23 +00:00)


In [ ]:
def process_chunk_of_data(lower_index, higher_index, download_dir = "/content"):
  base_path = download_dir + "/nlst"
  out_path = download_dir + "/nlst/features.csv"
  csv_filepath = download_dir + "/nlst/ct_header_info.csv"
  download_data(download_dir, lower_index, higher_index)
  prepare_for_pyradiomics(base_path)
  extract_features(base_path, out_path, interpolator='sitkBSpline')
  extract_ct_header_info(base_path, csv_filepath)
  delete_data(base_path)

time: 1.16 ms (started: 2026-04-07 15:32:41 +00:00)


In [ ]:
process_chunk_of_data(0,3)

Downloaded 12 series for 3 patients to Colab VM at /content...
Converted CT for patient 107002, timestep 1.2.840.113654.2.55.136072716603498717401162853674652254104 to /content/nlst/107002/1.2.840.113654.2.55.136072716603498717401162853674652254104
Converted SEG for patient 107002, timestep 1.2.840.113654.2.55.136072716603498717401162853674652254104 to 1.2.840.113654.2.55.136072716603498717401162853674652254104
Renamed masks for patient 107002, timestep 1.2.840.113654.2.55.136072716603498717401162853674652254104
Converted CT for patient 107002, timestep 1.2.840.113654.2.55.35671975265195223904685112987162619590 to /content/nlst/107002/1.2.840.113654.2.55.35671975265195223904685112987162619590
Converted SEG for patient 107002, timestep 1.2.840.113654.2.55.35671975265195223904685112987162619590 to 1.2.840.113654.2.55.35671975265195223904685112987162619590
Renamed masks for patient 107002, timestep 1.2.840.113654.2.55.35671975265195223904685112987162619590
Converted CT for patient 107030,

INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'resegmentRange': [-1000, 2000], 'label': 2, 'interpolator': 'sitkBSpline', 'resampledPixelSpacing': [1.0, 1.0, 1.0]}
INFO:radiomics.featureextractor:Loading image and mask


Converted SEG for patient 126955, timestep 1.2.840.113654.2.55.245452063932624888085703626412362172760 to 1.2.840.113654.2.55.245452063932624888085703626412362172760
Renamed masks for patient 126955, timestep 1.2.840.113654.2.55.245452063932624888085703626412362172760
Isotropic resampling to [1.0, 1.0, 1.0] using interpolator sitkBSpline
Extracting radiomic features...


INFO:radiomics.imageoperations:Applying resampling from spacing [0.603516 0.603516 2.5     ] and size [512 512 115] to spacing [1.  1.  2.5] and size [16, 17, 13]
INFO:radiomics.featureextractor:Computing shape
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Loading image and mask


Saving cropped nodule .nrrd file
Extracted features from 107002/1.2.840.113654.2.55.136072716603498717401162853674652254104


...

Extracted features from 107030/1.2.840.113654.2.55.75126321162854904399604743122014916460


INFO:radiomics.imageoperations:Applying resampling from spacing [0.703125 0.703125 2.      ] and size [512 512 132] to spacing [1. 1. 1.] and size [34, 37, 51]
INFO:radiomics.featureextractor:Computing shape
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Loading image and mask


Saving cropped nodule .nrrd file
Extracted features from 126955/1.2.840.113654.2.55.19724172053671692532719125746870801108


INFO:radiomics.imageoperations:Applying resampling from spacing [0.658203 0.658203 2.      ] and size [512 512 136] to spacing [1. 1. 1.] and size [33, 31, 27]
INFO:radiomics.featureextractor:Computing shape
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm


Saving cropped nodule .nrrd file
Extracted features from 126955/1.2.840.113654.2.55.245452063932624888085703626412362172760
Saved radiomic features to /content/features.csv
Reading CT from 107002/1.2.840.113654.2.55.136072716603498717401162853674652254104
Reading CT from 107002/1.2.840.113654.2.55.35671975265195223904685112987162619590
Reading CT from 107030/1.2.840.113654.2.55.279742797608242185321251439412013425352
Reading CT from 107030/1.2.840.113654.2.55.75126321162854904399604743122014916460
Reading CT from 126955/1.2.840.113654.2.55.19724172053671692532719125746870801108
Reading CT from 126955/1.2.840.113654.2.55.245452063932624888085703626412362172760
Saved CT header info to /content/ct_header_info.csv
time: 2min 12s (started: 2026-04-07 15:16:10 +00:00)


In [ ]:
process_chunk_of_data(4,7)

**IGNORE**

In [ ]:
from google.colab import files

files_to_download = [
    '/content/malignant_trial/features_manual_resampling.csv',
    '/content/malignant_trial/features_pyradiomics_resampling.csv',
]

for file_path in files_to_download:
    try:
        files.download(file_path)
        print(f"Successfully initiated download for {file_path}")
    except Exception as e:
        print(f"Failed to download {file_path}: {e}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Successfully initiated download for /content/malignant_trial/features_manual_resampling.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Successfully initiated download for /content/malignant_trial/features_pyradiomics_resampling.csv
time: 11 ms (started: 2026-03-31 13:14:17 +00:00)


In [ ]:
# manual resampling
# prepare_for_pyradiomics('/content/malignant_trial/nlst', resample=True)

Converted CT for patient 126955, timestep 1.2.840.113654.2.55.19724172053671692532719125746870801108 to /content/malignant_trial/nlst/126955/1.2.840.113654.2.55.19724172053671692532719125746870801108
Converted SEG for patient 126955, timestep 1.2.840.113654.2.55.19724172053671692532719125746870801108 to 1.2.840.113654.2.55.19724172053671692532719125746870801108
Renamed masks for patient 126955, timestep 1.2.840.113654.2.55.19724172053671692532719125746870801108
Resampled and aligned CT and masks for patient 126955, timestep 1.2.840.113654.2.55.19724172053671692532719125746870801108 to 1.2.840.113654.2.55.19724172053671692532719125746870801108
Plotted CT with masks for patient 126955, timestep 1.2.840.113654.2.55.19724172053671692532719125746870801108
Converted CT for patient 126955, timestep 1.2.840.113654.2.55.245452063932624888085703626412362172760 to /content/malignant_trial/nlst/126955/1.2.840.113654.2.55.245452063932624888085703626412362172760
Converted SEG for patient 126955, tim

In [ ]:
# base_path = '/content/malignant_trial/nlst'
# out_path = '/content/malignant_trial/features_manual_resampling.csv'
# extract_features(base_path, out_path, resample=False)

INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'resegmentRange': [-1000, 2000], 'label': 2}
INFO:radiomics.featureextractor:Calculating features with label: 2
INFO:radiomics.featureextractor:Loading image and mask


Extracting radiomic features...


INFO:radiomics.featureextractor:Computing shape
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 2
INFO:radiomics.featureextractor:Loading image and mask


Extracted features from 126955/1.2.840.113654.2.55.19724172053671692532719125746870801108


INFO:radiomics.featureextractor:Computing shape
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 2
INFO:radiomics.featureextractor:Loading image and mask


...

Extracted features from 107002/1.2.840.113654.2.55.35671975265195223904685112987162619590


INFO:radiomics.featureextractor:Computing shape
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 2
INFO:radiomics.featureextractor:Loading image and mask


Extracted features from 107002/1.2.840.113654.2.55.136072716603498717401162853674652254104


INFO:radiomics.featureextractor:Computing shape
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm


Extracted features from 126446/1.2.840.113654.2.55.257572809223995827394443746663383442894
Saved radiomic features to /content/malignant_trial/features_manual_resampling.csv
time: 2min 38s (started: 2026-03-31 13:03:12 +00:00)


In [ ]:
%pip install pydicom
import pydicom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.4 MB/s eta 0:00:00
time: 11.5 s (started: 2026-03-29 22:59:17 +00:00)


In [ ]:
# maybe don't do this for the larger dataset? Just assume SEG_1 is lung and SEG_2 is nodule?
def identify_and_rename_masks_using_pydicom(seg_file, timestep_path, prefix="SEG"):
  ds = pydicom.dcmread(seg_file)
  segment_labels = []
  for segment in ds.SegmentSequence:
    label = segment.SegmentLabel.lower().replace(" ", "_")
    segment_labels.append(label)
  for i, label in enumerate(segment_labels):
    segment_number = i+1
    old_name = os.path.join(timestep_path, f"{prefix}-{segment_number}.nrrd")
    new_name = os.path.join(timestep_path, f"{label}.nrrd")
    if os.path.exists(old_name):
      os.rename(old_name, new_name)
    else:
      print(f"File not found: {old_name}")

time: 845 µs (started: 2026-03-29 22:59:28 +00:00)


In [ ]:
def identify_and_rename_masks(seg_file, timestep_path, prefix="SEG"):
  json_files = glob.glob(os.path.join(timestep_path, f"{prefix}-*.json"))
  if not json_files:
    identify_and_rename_masks_using_pydicom(seg_file, timestep_path, prefix)
  else:
    with open(json_files[0], 'r') as f:
      metadata = json.load(f)
      for i, segment in enumerate(metadata.get('segments', [])):
        label = segment.get('SegmentLabel', '').lower()
        segment_number = i+1
        old_name = os.path.join(timestep_path, f"{prefix}-{segment_number}.nrrd")
        if "lung" in label:
          new_name = os.path.join(timestep_path, "lung.nrrd")
          os.rename(old_name, new_name)
        elif "nodule" in label:
          new_name = os.path.join(timestep_path, "nodule.nrrd")
          os.rename(old_name, new_name)
    os.remove(json_files[0])

time: 1.11 ms (started: 2026-03-29 22:59:28 +00:00)


In [ ]:
def prepare_for_pyradiomics_error_handling(base_path, align_and_plot_masks=True):
  for patient_id in os.listdir(base_path):
    patient_path = os.path.join(base_path, patient_id)
    if not os.path.isdir(patient_path):
        continue

    for timestep_dir in os.listdir(patient_path):
        timestep_path = os.path.join(patient_path, timestep_dir)
        if not os.path.isdir(timestep_path):
            continue

        ct_series_path = None
        seg_series_path = None

        for item in os.listdir(timestep_path):
            full_item_path = os.path.join(timestep_path, item)
            if os.path.isdir(full_item_path):
                if item.startswith('CT_'):
                    ct_series_path = full_item_path
                elif item.startswith('SEG_'):
                    seg_series_path = full_item_path

        if ct_series_path:
            try:
                convert_CT_to_nrrd(ct_series_path, timestep_path)
                print(f"Converted CT for patient {patient_id}, timestep {timestep_dir} to {timestep_path}")
            except Exception as e:
                print(f"Failed to convert CT for patient {patient_id}, timestep {timestep_dir}: {e}")
        else:
            print(f"CT series folder not found for patient {patient_id}, timestep {timestep_dir}. Skipping CT and SEG conversion for this timestep.")
            continue # Skip to next timestep if CT is not found

        if seg_series_path:
            try:
                seg_dcm_files = [f for f in os.listdir(seg_series_path) if f.endswith('.dcm')] # should be only one .dcm always, so remove this?
                if not seg_dcm_files:
                    print(f"No DICOM SEG files found in SEG series for patient {patient_id}, timestep {timestep_dir}")
                    continue
                seg_file = os.path.join(seg_series_path, seg_dcm_files[0])
                if convert_SEG_to_nrrd(seg_file, timestep_path):
                  print(f"Converted SEG for patient {patient_id}, timestep {timestep_dir} to {timestep_dir}")
                  identify_and_rename_masks_assuming_naming_convention(timestep_path)
                  print(f"Renamed masks for patient {patient_id}, timestep {timestep_dir}")
                  if align_and_plot_masks:
                    resample_isotropic_and_align(timestep_path)
                    print(f"Resamples and aligned CT and masks for patient {patient_id}, timestep {timestep_dir} to {timestep_dir}")
                    plot_ct_with_masks(patient_id, timestep_dir, timestep_path)
                    print(f"Plotted CT with masks for patient {patient_id}, timestep {timestep_dir}")
            except Exception as e:
                print(f"Failed to convert SEG for patient {patient_id}, timestep {timestep_dir}: {e}")
        else:
            print(f"SEG series folder not found for patient {patient_id}, timestep {timestep_dir}")

time: 3.7 ms (started: 2026-03-29 22:59:40 +00:00)
